# Smartphone Usage and Addiction Analysis
## Setup and Data Loading

* Importing libraries 
* Loading the raw dataset 
* Doing a quick initial inspection.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')

display(df_raw.head())

print(f"Dataset Shape: {df_raw.shape}")

print("Column Names:")
print(df_raw.columns.tolist())

,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


Dataset Shape: (7500, 16)
Column Names:
['transaction_id', 'user_id', 'age', 'gender', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'stress_level', 'academic_work_impact', 'addiction_level', 'addicted_label']


## Cleaning and Data Quality

 Check data quality, inspect ranges, clean the gender column to strictly binary (Male/Female), and create the df_clean dataframe.

In [2]:
print(f"Shape before cleaning: {df_raw.shape}")

df_raw.columns = df_raw.columns.str.strip()

# Check data types
print("\n--- Data Types ---")
df_raw.info()

# Check missing values
print("\n--- Missing Values ---")
print(df_raw.isnull().sum())

# Check duplicates
print("\n--- Duplicate Checks ---")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
print(f"Duplicate transaction_ids: {df_raw['transaction_id'].duplicated().sum()}")
print(f"Duplicate user_ids: {df_raw['user_id'].duplicated().sum()}")

# Check unique values at categorical columns
categorical_cols = ['gender', 'stress_level', 'academic_work_impact', 'addiction_level']
print("\n--- Categorical Unique Values ---")
for col in categorical_cols:
    if col in df_raw.columns:
        # Strip spaces from categorical values to fix simple formatting issues
        if df_raw[col].dtype == 'object':
            df_raw[col] = df_raw[col].str.strip()
        print(f"{col}: {df_raw[col].unique()}")

# Check basic numeric ranges
print("\n--- Numeric Ranges ---")
display(df_raw.describe())

Shape before cleaning: (7500, 16)

--- Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           7500 non-null   object 
 1   user_id                  7500 non-null   object 
 2   age                      7500 non-null   int64  
 3   gender                   7500 non-null   object 
 4   daily_screen_time_hours  7500 non-null   float64
 5   social_media_hours       7500 non-null   float64
 6   gaming_hours             7500 non-null   float64
 7   work_study_hours         7500 non-null   float64
 8   sleep_hours              7500 non-null   float64
 9   notifications_per_day    7500 non-null   int64  
 10  app_opens_per_day        7500 non-null   int64  
 11  weekend_screen_time      7500 non-null   float64
 12  stress_level             7500 non-null   object 
 13  academic_work_impact    

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addicted_label
count,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,26.568800,7.499912,3.273484,2.014183,3.242420,6.737561,134.257333,97.832400,9.243827,0.707733
std,5.197108,2.609188,1.585342,1.146039,1.600765,1.283605,66.586883,48.423349,2.718281,0.454835
min,18.000000,3.000000,0.500000,0.000000,0.500000,4.500000,20.000000,15.000000,3.580000,0.000000
25%,22.000000,5.220000,1.910000,1.020000,1.850000,5.630000,76.000000,55.000000,6.960000,0.000000
50%,27.000000,7.525000,3.270000,2.040000,3.230000,6.720000,134.000000,98.000000,9.260000,1.000000
75%,31.000000,9.810000,4.630000,2.990000,4.640000,7.840000,191.000000,140.000000,11.540000,1.000000
max,35.000000,12.000000,6.000000,4.000000,6.000000,9.000000,250.000000,180.000000,14.880000,1.000000


In [3]:
# Clean gender as a binary field
df_clean = df_raw.copy()

# Standardize
gender_mapping = {
    'male': 'Male', 'Male': 'Male', 'M': 'Male',
    'female': 'Female', 'Female': 'Female', 'F': 'Female'
}

df_clean['gender'] = df_clean['gender'].replace(gender_mapping)

# Find rows where gender is NOT 'Male' or 'Female'
valid_genders = ['Male', 'Female']
invalid_gender_mask = ~df_clean['gender'].isin(valid_genders)
invalid_gender_count = invalid_gender_mask.sum()

# Remove rows
df_clean = df_clean[~invalid_gender_mask]

print(f"Rows removed due to non-binary/invalid gender: {invalid_gender_count}")
print(f"Final unique genders in df_clean: {df_clean['gender'].unique()}")

print(f"\nFinal df_clean shape: {df_clean.shape}")

Rows removed due to non-binary/invalid gender: 2486
Final unique genders in df_clean: ['Male' 'Female']

Final df_clean shape: (5014, 16)


### Person 1 Handoff to Person 2

1. Rows and columns after cleaning: 5014 rows and 16 columns.
2. Missing values still exist in `addiction_level`: 547 missing values after removing invalid gender rows. No other columns have missing values.
3. Duplicate rows: 0. Duplicate `transaction_id`: 0. Duplicate `user_id`: 0.
4. Final unique values in `gender`: `Male` and `Female`.
5. Rows removed because `gender` was not `Male` or `Female`: 2486 rows with `Other` gender values.
6. Unusual values found: `Other` appeared in `gender` and was removed because this project uses binary gender only. `addiction_level` contains missing values, which should be handled in the preprocessing phase.
7. `df_clean` is ready for Person 2 to continue preprocessing and feature preparation.


## 2. Preprocessing and Feature Preparation - Person 2

This section starts from `df_clean`, handles the remaining missing values, prepares useful engineered features, encodes `gender` for modeling, and creates `df_prepared`, `X`, and `y` for the next phases.

Important modeling rule: `transaction_id`, `user_id`, `addicted_label`, `addiction_level`, and the original text `gender` column are excluded from the feature matrix. `addiction_level` is excluded because it is closely related to the target and may leak target information.

In [ ]:
# Start from the cleaned dataframe created by Person 1
df_prepared = df_clean.copy()

numeric_columns = [
    'age',
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time',
    'addicted_label'
]

categorical_columns = [
    'gender',
    'stress_level',
    'academic_work_impact',
    'addiction_level'
]

identifier_columns = ['transaction_id', 'user_id']

print('Missing values before preprocessing:')
display(df_prepared.isna().sum().to_frame('missing_count'))

# Do not impute the target. If target values are missing, remove those rows.
target_missing = df_prepared['addicted_label'].isna().sum()
if target_missing > 0:
    df_prepared = df_prepared.dropna(subset=['addicted_label']).copy()

# Numeric columns use median imputation if missing values are present.
numeric_imputation_values = {}
for col in numeric_columns:
    if col != 'addicted_label' and df_prepared[col].isna().sum() > 0:
        median_value = df_prepared[col].median()
        df_prepared[col] = df_prepared[col].fillna(median_value)
        numeric_imputation_values[col] = median_value

# Categorical columns use a simple category for missing values.
# For addiction_level, Unknown preserves the fact that the original value was missing.
categorical_imputation_values = {}
for col in categorical_columns:
    if col != 'gender' and df_prepared[col].isna().sum() > 0:
        fill_value = 'Unknown'
        df_prepared[col] = df_prepared[col].fillna(fill_value)
        categorical_imputation_values[col] = fill_value

# Confirm gender is still binary before encoding.
valid_genders = ['Female', 'Male']
remaining_invalid_gender_count = (~df_prepared['gender'].isin(valid_genders)).sum()
print(f'Remaining invalid gender values: {remaining_invalid_gender_count}')

gender_binary_map = {'Female': 0, 'Male': 1}
df_prepared['gender_binary'] = df_prepared['gender'].map(gender_binary_map).astype(int)

print('\nNumeric columns:')
print(numeric_columns)

print('\nCategorical columns:')
print(categorical_columns)

print('\nNumeric imputation values used:')
print(numeric_imputation_values if numeric_imputation_values else 'No numeric imputation needed')

print('\nCategorical imputation values used:')
print(categorical_imputation_values if categorical_imputation_values else 'No categorical imputation needed')

print('\nMissing values after preprocessing:')
display(df_prepared.isna().sum().to_frame('missing_count'))

print('\nGender encoding check:')
display(df_prepared[['gender', 'gender_binary']].drop_duplicates().sort_values('gender_binary'))

### Feature Preparation

The original cleaned columns are kept in `df_prepared` for EDA. New features are added to summarize usage behavior and create intensity measures. Ratio features safely handle division by zero by replacing invalid ratios with `0`.

`gender_binary` uses this encoding: `Female = 0`, `Male = 1`. The original `gender` column is kept for charts and grouped summaries, but it is not used directly in the modeling matrix.

In [ ]:
# Simple engineered features
df_prepared['usage_component_total_hours'] = (
    df_prepared['social_media_hours']
    + df_prepared['gaming_hours']
    + df_prepared['work_study_hours']
)

screen_time_nonzero = df_prepared['daily_screen_time_hours'].replace(0, np.nan)

df_prepared['notifications_per_screen_hour'] = (
    df_prepared['notifications_per_day'] / screen_time_nonzero
).replace([np.inf, -np.inf], np.nan).fillna(0)

df_prepared['app_opens_per_screen_hour'] = (
    df_prepared['app_opens_per_day'] / screen_time_nonzero
).replace([np.inf, -np.inf], np.nan).fillna(0)

df_prepared['sleep_below_7_hours'] = (df_prepared['sleep_hours'] < 7).astype(int)

excluded_modeling_columns = [
    'transaction_id',
    'user_id',
    'addicted_label',
    'addiction_level',
    'gender'
]

numeric_model_features = [
    'age',
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time',
    'gender_binary',
    'usage_component_total_hours',
    'notifications_per_screen_hour',
    'app_opens_per_screen_hour',
    'sleep_below_7_hours'
]

categorical_model_features = ['stress_level', 'academic_work_impact']
model_feature_columns = numeric_model_features + categorical_model_features

# Keep an unencoded feature dataframe available for sklearn ColumnTransformer pipelines.
X_unencoded = df_prepared[model_feature_columns].copy()

# Create a fully numeric feature matrix for simple models.
X = pd.get_dummies(
    X_unencoded,
    columns=categorical_model_features,
    drop_first=False,
    dtype=int
)

y = df_prepared['addicted_label'].astype(int).copy()

print(f'df_prepared shape: {df_prepared.shape}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

print('\nExcluded modeling columns:')
print(excluded_modeling_columns)

print('\nRecommended unencoded feature columns for Person 4 pipeline option:')
print(model_feature_columns)

print('\nFinal numeric X columns:')
print(X.columns.tolist())

display(df_prepared.head())
display(X.head())
display(y.value_counts().sort_index().rename('target_count').to_frame())

### Person 2 Handoff to Person 3 and Person 4

1. Preprocessing started from `df_clean`, which already removed non-binary `gender` values according to the project rule.
2. Missing `addiction_level` values were filled with `Unknown` in `df_prepared` so Person 3 can still include those rows in EDA. This column should still be excluded from modeling because it may leak target information.
3. No target imputation was applied to `addicted_label`.
4. `gender` was encoded as `gender_binary`, where `Female = 0` and `Male = 1`. The original `gender` column remains available for EDA labels.
5. New features created: `usage_component_total_hours`, `notifications_per_screen_hour`, `app_opens_per_screen_hour`, and `sleep_below_7_hours`.
6. Columns excluded from modeling: `transaction_id`, `user_id`, `addicted_label`, `addiction_level`, and `gender`.
7. `df_prepared` is ready for Person 3. `X`, `X_unencoded`, `y`, `model_feature_columns`, and `excluded_modeling_columns` are ready for Person 4.
